# [0] Phase 1 - Clean VQA with TALE-EP
**Images**: image1.png, image2.png, ...  
**Task**: English prompt optimization for proceed-straight VQA  
**Env**: Google Colab T4, Qwen2.5-VL 3B 4bit  
**Method**: TALE-EP prompt candidate optimization


In [ ]:
# [1] 환경 확인
import os

# CUDA allocator 설정은 torch import 전에 적용되어야 합니다.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch


IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('환경:', 'Colab' if IN_COLAB else 'Local')
print('device:', DEVICE)

if DEVICE == 'cuda':
    pass
elif DEVICE == 'mps':
    print('MPS (Apple Silicon) 환경입니다. bfloat16으로 로드합니다.')
else:
    print('경고: GPU 없음. CPU 추론은 매우 느릴 수 있습니다.')


In [ ]:
# [2] 패키지 설치
if IN_COLAB:
    %pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils


In [ ]:
# [3] Image upload and inference resize
from pathlib import Path

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / 'data').is_dir() and (cand / 'experiments').is_dir():
            return cand
    return start
PROJECT_ROOT = find_repo_root()
import re
from PIL import Image
from IPython.display import display, Image as IPImage

MAX_IMAGE_SIDE = 896
SUPPORTED_IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}

if IN_COLAB:
    from google.colab import files
    print('Upload one or more image files, e.g. image1.png, image2.png.')
    uploaded = files.upload()
    IMAGE_ORIGINALS = [
        name for name in uploaded.keys()
        if Path(name).suffix.lower() in SUPPORTED_IMAGE_EXTS
    ]
else:
    raw_dir = PROJECT_ROOT / 'data/raw'
    image_name_pattern = re.compile(r'^image\d+$', re.IGNORECASE)
    IMAGE_ORIGINALS = [
        str(p) for p in sorted(raw_dir.iterdir())
        if p.suffix.lower() in SUPPORTED_IMAGE_EXTS and image_name_pattern.match(p.stem)
    ]

if not IMAGE_ORIGINALS:
    raise FileNotFoundError('No supported image files found. Use names like image1.png, image2.png, or image123.png.')

IMAGES = []
for idx, image_original in enumerate(IMAGE_ORIGINALS, start=1):
    src = Path(image_original)
    img = Image.open(src).convert('RGB')
    img.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS)
    infer_name = f'{src.stem}_infer.png'
    img.save(infer_name)
    IMAGES.append({
        'original': str(image_original),
        'infer': infer_name,
        'stem': src.stem,
        'size': img.size,
    })
    print(f'[{idx}] original={image_original}  infer={infer_name}  size={img.size}')
    display(IPImage(filename=infer_name, width=640))

print('Images ready:', [item['stem'] for item in IMAGES])


In [ ]:
# [4] 모델 로드
import torch
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, AutoProcessor

# T4 OOM을 줄이기 위해 기본값은 3B입니다. 7B가 필요하면 아래 값을 7B로 바꾸고 이미지 크기를 더 낮추세요.
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
# MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

if DEVICE == 'cuda':
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map='auto',
        attn_implementation='eager',
    )
elif DEVICE == 'mps':
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        attn_implementation='eager',
    ).to('mps')
else:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32,
        device_map='cpu',
        attn_implementation='eager',
    )

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=224 * 224,
    max_pixels=896 * 896,
)
print(f'모델 로드 완료: {MODEL_ID}')


In [ ]:
# [5] Prompt candidates / TALE-EP settings - edit prompts here
# To change prompt candidates, edit PROMPT_CANDIDATES.
# To change output budget candidates, edit TALE_EP_BUDGET_CANDIDATES.
# All prompts request English answers.

PROMPT_CANDIDATES = [
  {
    "id": "prompt_01",
    "style": "concise",
    "prompt": "You are an autonomous-driving assistance evaluator. Use only information directly visible in the image to decide whether the vehicle may proceed straight.\n\nOutput format:\nDecision: [Proceed / Do not proceed / Cannot determine]\nEvidence:\n- Observation: ... -> Impact: ...\n- Observation: ... -> Impact: ...\n\nRules:\n- Use only traffic lights, signs, pedestrians, vehicle flow, and obstacles as evidence.\n- Do not infer situations that are not visible in the image.\n- Include all directly visible safety-critical cues; do not omit relevant evidence only to be brief.\n- If uncertain, choose 'Cannot determine'.",
    "expected_strength": "Compact structure while allowing all visible safety-critical evidence.",
    "risk": "May still compress details too aggressively in complex scenes."
  },
  {
    "id": "prompt_02",
    "style": "safety_focused",
    "prompt": "Look at the forward road image and decide from a safety perspective whether the vehicle may proceed straight.\n\nOutput:\nDecision: [Proceed / Do not proceed / Cannot determine]\nSafety evidence:\n1. Observed element: ...\n   Impact on decision: ...\n2. Observed element: ...\n   Impact on decision: ...\n\nRules:\n- Use only visible objects.\n- If a visible risk signal exists, judge conservatively.\n- If visibility or evidence is insufficient, choose 'Cannot determine'.\n- Avoid unnecessary explanation.",
    "expected_strength": "Strongly encourages safety-first reasoning and risk detection.",
    "risk": "May become overly conservative in scenes where proceeding is allowed."
  },
  {
    "id": "prompt_03",
    "style": "evidence_strict",
    "prompt": "Decide whether the vehicle may proceed straight using only evidence directly visible inside the image.\n\nOutput format:\n{\n  \"decision\": \"Proceed | Do not proceed | Cannot determine\",\n  \"evidence\": [\n    {\n      \"observation\": \"...\",\n      \"impact\": \"...\"\n    }\n  ]\n}\n\nRules:\n- No guessing or filling gaps with general knowledge.\n- Do not mention objects that are not visually observable.\n- If the evidence is weak, choose 'Cannot determine'.",
    "expected_strength": "Good for evidence grounding and hallucination control.",
    "risk": "May produce more 'Cannot determine' answers than necessary."
  },
  {
    "id": "prompt_04",
    "style": "uncertainty_aware",
    "prompt": "Given one road image, decide whether the vehicle may proceed straight. If visibility, occlusion, or resolution makes the judgment uncertain, defer the decision.\n\nOutput:\nDecision: [Proceed / Do not proceed / Cannot determine]\nConfidence: [High / Medium / Low]\nEvidence:\n- Observation: ... -> Impact: ...\n\nRules:\n- Use only visible information.\n- Include every directly visible cue that affects the decision, while keeping each item concise.\n- Do not force a decision.",
    "expected_strength": "Captures uncertainty and is useful for comparing robust early-exit behavior.",
    "risk": "Confidence reporting may increase token usage slightly."
  },
  {
    "id": "prompt_05",
    "style": "token_efficient",
    "prompt": "Analyze the forward-driving image.\n\nFormat:\nDecision: [Proceed / Do not proceed / Cannot determine]\nEvidence:\n- Observation -> Impact\n\nRules:\n- Keep the answer concise, but do not omit safety-critical evidence.\n- Use only directly visible objects.\n- Do not guess.",
    "expected_strength": "Concise format useful for token and latency optimization without a fixed evidence count.",
    "risk": "May still under-explain if the model compresses too aggressively."
  },
  {
    "id": "prompt_06",
    "style": "evidence_strict",
    "prompt": "Judge whether the vehicle may proceed straight based only on visual evidence in the image.\n\nStrict format:\nDecision: ...\nEvidence:\n- [Object] observed state / impact on decision\n\nAllowed objects:\ntraffic lights, signs, pedestrians, vehicles, obstacles, lane state\n\nForbidden:\n- Predicting future events\n- Inferring driver intent\n- Deciding from general traffic rules alone",
    "expected_strength": "Limits evidence scope and improves grounding.",
    "risk": "May ignore useful cues outside the allowed object list."
  },
  {
    "id": "prompt_07",
    "style": "safety_focused",
    "prompt": "You are a conservative autonomous-driving safety evaluator. If a visible risk for proceeding straight exists, choose 'Do not proceed' or 'Cannot determine'.\n\nOutput:\nDecision: [Proceed / Do not proceed / Cannot determine]\nRisk evidence:\n- Observation: ...\n  Impact: ...\n\nConditions:\n- Mention only visible risks.\n- Choose Proceed only when no risk factor is visible.\n- Keep the answer short.",
    "expected_strength": "Helps reduce false negatives in safety-critical scenes.",
    "risk": "May increase false positives."
  },
  {
    "id": "prompt_08",
    "style": "uncertainty_aware",
    "prompt": "Determine whether the vehicle may proceed straight from the forward road image.\n\nOutput format:\nConclusion: [Proceed / Do not proceed / Cannot determine]\nEvidence:\n1. Observed fact\n2. How that fact affects the straight-driving decision\n\nDecision rules:\n- Do not use information that is not directly visible.\n- If evidence is insufficient, choose Cannot determine.\n- Make the evidence and conclusion logically connected.",
    "expected_strength": "Strongly enforces connection between observation and judgment.",
    "risk": "May use more tokens because of the structured reasoning requirement."
  },
  {
    "id": "prompt_09",
    "style": "token_efficient",
    "prompt": "Look at the road image and judge only whether the vehicle may proceed straight.\n\nOutput:\nDecision: [Proceed / Do not proceed / Cannot determine]\nEvidence: Observation -> Impact\n\nRules:\n- Use only what is visible in the image.\n- Avoid long explanations.\n- If uncertain, choose Cannot determine.",
    "expected_strength": "Concise while allowing multiple safety-critical cues when needed.",
    "risk": "May become brief unless safety-critical cue inclusion is followed."
  },
  {
    "id": "prompt_10",
    "style": "concise",
    "prompt": "Perform an autonomous-driving judgment.\n\nFirst select only the key objects relevant to proceeding straight. Then decide whether proceeding straight is allowed.\n\nOutput format:\nKey observations:\n- ...\nDecision: [Proceed / Do not proceed / Cannot determine]\nReason: ...\n\nConstraints:\n- Include all key safety-critical observations.\n- Use only directly observed information.\n- Do not speculate.",
    "expected_strength": "Encourages selecting all safety-critical objects before the final judgment.",
    "risk": "The model may still omit cues if it judges them non-critical."
  }
]

TALE_EP_BUDGET_CANDIDATES = [64, 80, 96, 120, 144]

def build_tale_ep_prompt(prompt: str, budget: int) -> str:
    constraint = f"""

TALE-EP output budget: at most {budget} tokens.
Answer in English only. Compress the key safety objects and decision rationale, but include both observation and impact."""
    return prompt + constraint

def quality_check(output: str) -> dict:
    lower = output.lower()
    has_decision = any(label in lower for label in ['proceed', 'do not proceed', 'cannot determine'])
    has_reason = any(term in lower for term in ['evidence', 'reason', 'observation', 'observed', 'impact'])
    has_impact = any(term in lower for term in ['impact', 'because', 'therefore', 'risk', 'allows', 'prevents', 'indicates'])
    visible_object_terms = ['traffic light', 'sign', 'pedestrian', 'obstacle', 'vehicle', 'lane', 'green', 'red', 'intersection']
    object_mentions = [term for term in visible_object_terms if term in lower]
    hallucination_risk_terms = ['probably', 'seems like', 'not visible but', 'generally', 'guess', 'assume']
    hallucination_risk = any(term in lower for term in hallucination_risk_terms)
    non_english_markers = ['판단', '근거', '관찰', '영향', '직진']
    has_non_english = any(term in output for term in non_english_markers)
    return {
        'has_decision': has_decision,
        'has_reason': has_reason,
        'has_impact': has_impact,
        'object_mentions': object_mentions,
        'hallucination_risk': hallucination_risk,
        'has_non_english': has_non_english,
        'quality_score': (
            int(has_decision)
            + int(has_reason)
            + int(has_impact)
            + min(len(object_mentions), 3)
            - int(hallucination_risk)
            - int(has_non_english)
        ),
    }

def optimization_score(result: dict) -> float:
    # Prefer high quality; break ties with lower input/output tokens and lower latency.
    quality = result['quality']['quality_score']
    input_penalty = result['input_tokens'] / 1000
    output_penalty = result['output_tokens'] / 100
    latency_penalty = result['latency_sec'] / 20
    return round(quality - input_penalty - output_penalty - latency_penalty, 4)


In [ ]:
# [6] Inference function
import gc
import time
from qwen_vl_utils import process_vision_info

def infer(image_path: str, prompt: str, max_new_tokens: int) -> dict:
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE == 'mps':
        torch.mps.empty_cache()
    gc.collect()

    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image_path},
        {'type': 'text', 'text': prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors='pt')
    input_tokens = int(inputs['input_ids'].shape[1])
    inputs = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

    t0 = time.time()
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    latency = round(time.time() - t0, 2)

    out_ids = generated[0][inputs['input_ids'].shape[1]:]
    output = processor.decode(out_ids, skip_special_tokens=True).strip()
    output_tokens = int(len(out_ids))
    return {
        'output': output,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': input_tokens + output_tokens,
        'latency_sec': latency,
    }


In [ ]:
# [7] Run prompt candidates x output budgets for each image with pruning
import json

results = {}
best = None

MIN_GOOD_QUALITY = 6
COMPLETION_MARGIN = 0.90
MAX_WORSE_STREAK = 2


def looks_truncated(output: str, output_tokens: int, budget: int) -> bool:
    stripped = output.strip()
    if output_tokens >= budget:
        return True
    if not stripped:
        return True
    if stripped[-1] not in '.]}':
        return True
    dangling_endings = (' to', ' of', ' and', ' the', ' a', ' an', ' Ign', ' before', ' with')
    return stripped.endswith(dangling_endings)


for image_item in IMAGES:
    image_key = image_item['stem']
    results[image_key] = {}
    print('\n' + '#' * 70)
    print(f'[IMAGE] {image_key} ({image_item["infer"]})')

    for candidate in PROMPT_CANDIDATES:
        candidate_best = None
        worse_streak = 0

        for budget in TALE_EP_BUDGET_CANDIDATES:
            name = f'{candidate["id"]}_budget_{budget}'
            print(f'\n{"-" * 50}\n[{image_key} / {name}] style={candidate["style"]}')
            final_prompt = build_tale_ep_prompt(candidate['prompt'], budget)
            r = infer(image_item['infer'], final_prompt, budget)
            r.update({
                'image_original': image_item['original'],
                'image_infer': image_item['infer'],
                'candidate_id': candidate['id'],
                'style': candidate['style'],
                'prompt': candidate['prompt'],
                'final_prompt': final_prompt,
                'expected_strength': candidate['expected_strength'],
                'risk': candidate['risk'],
                'predicted_output_tokens': budget,
                'token_budget_error': r['output_tokens'] - budget,
                'quality': quality_check(r['output']),
            })
            r['optimization_score'] = optimization_score(r)
            r['truncated'] = looks_truncated(r['output'], r['output_tokens'], budget)
            r['pruned_after_this'] = False
            r['prune_reason'] = ''
            results[image_key][name] = r

            overall_candidate = {'image': image_key, 'name': name, **r}
            if best is None or r['optimization_score'] > best['optimization_score']:
                best = overall_candidate

            if candidate_best is None or r['optimization_score'] > candidate_best['optimization_score']:
                candidate_best = {'image': image_key, 'name': name, **r}
                worse_streak = 0
            else:
                worse_streak += 1

            print(r['output'])
            print(
                f'\nscore={r["optimization_score"]}  '
                f'quality={r["quality"]["quality_score"]}  '
                f'input={r["input_tokens"]}  '
                f'output={r["output_tokens"]}/{budget}  '
                f'total={r["total_tokens"]}  '
                f'truncated={r["truncated"]}  '
                f'latency={r["latency_sec"]}s'
            )

            good_complete_answer = (
                r['quality']['quality_score'] >= MIN_GOOD_QUALITY
                and not r['truncated']
                and r['output_tokens'] <= budget * COMPLETION_MARGIN
            )
            dominated_by_smaller_budget = (
                candidate_best is not None
                and candidate_best['name'] != name
                and r['quality']['quality_score'] <= candidate_best['quality']['quality_score']
                and r['optimization_score'] < candidate_best['optimization_score']
                and r['output_tokens'] >= candidate_best['output_tokens']
            )

            if good_complete_answer:
                r['pruned_after_this'] = True
                r['prune_reason'] = 'good_complete_answer'
                print(f'[PRUNE] Stop larger budgets for {candidate["id"]}: good complete answer at budget={budget}.')
                break

            if dominated_by_smaller_budget and worse_streak >= MAX_WORSE_STREAK:
                r['pruned_after_this'] = True
                r['prune_reason'] = 'dominated_by_smaller_budget'
                print(f'[PRUNE] Stop larger budgets for {candidate["id"]}: larger budgets are dominated by smaller budget result.')
                break

print('\n' + '=' * 50)
print('[BEST OVERALL]')
print('image:', best['image'])
print('name:', best['name'])
print('style:', best['style'])
print('budget:', best['predicted_output_tokens'])
print('input_tokens:', best['input_tokens'])
print('output_tokens:', best['output_tokens'])
print('total_tokens:', best['total_tokens'])
print('score:', best['optimization_score'])
print('prompt:\n', best['prompt'])
print('output:\n', best['output'])


In [ ]:
# [8] Save results
manual = {
    'label': '',  # 'Proceed', 'Do not proceed', or 'Cannot determine'
    'safety_objects': [],
    'notes': '',
}

out = {
    'images': IMAGES,
    'model': MODEL_ID,
    'device': DEVICE,
    'method': 'TALE-EP prompt candidate optimization',
    'prompt_candidates': PROMPT_CANDIDATES,
    'budget_candidates': TALE_EP_BUDGET_CANDIDATES,
    'best': best,
    'manual_label': manual,
    'results': results,
}

if IN_COLAB:
    from google.colab import files
    out_file = 'phase1_prompt_optimization_results.json'
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    files.download(out_file)
else:
    out_path = PROJECT_ROOT / 'experiments/results/phase1_prompt_optimization_results.json'
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f'Saved: {out_path}')
